# HUẤN LUYỆN MÔ HÌNH VÀ THỬ NGHIỆM THAM SỐ

Đề tài: Phân loại cảm xúc từ phản hồi của sinh viên (FIT4013).
Đầu vào: results/member2_outputs/ do notebook 02 tạo.
Đầu ra: results/training/ gồm thống kê CV, bảng đánh giá trên test, dự đoán theo mẫu, và mô hình đã lưu.

Nguyên tắc: Không đọc/chia lại CSV gốc; giữ test cố định; GridSearchCV chỉ chạy trên train và vectorizer
được fit độc lập theo từng fold. Chỉ sử dụng test sau khi đã chọn xong siêu tham số bằng CV.

## 1. Nạp thư viện và xác định đường dẫn dự án

In [1]:
from pathlib import Path
import json, joblib, numpy as np, pandas as pd
from IPython.display import display
from scipy.sparse import load_npz
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,classification_report)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd,*cwd.parents] if (p/'results'/'member2_outputs'/'experiment_config.json').is_file()),None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cần chạy 02_Preprocessing.ipynb để tạo results/member2_outputs trước.')
IN_DIR = PROJECT_ROOT/'results'/'member2_outputs'
OUT_DIR = PROJECT_ROOT/'results'/'training'
MODEL_DIR = OUT_DIR/'models'
OUT_DIR.mkdir(parents=True,exist_ok=True)
MODEL_DIR.mkdir(parents=True,exist_ok=True)
config = json.loads((IN_DIR/'experiment_config.json').read_text(encoding='utf-8'))
RANDOM_STATE = int(config['random_state'])
LABEL_COL = config['target_column']
print('Nhận dữ liệu từ:',IN_DIR)


Nhận dữ liệu từ: D:\Nam3\MachineLearning\Nhom-7_Phan_loai_cam_xuc_tu_phan_hoi_cua_sinh_vien-main\results\member2_outputs


## 2. Đọc và xác thực train/test từ 02_Preprocessing

In [2]:
train_df = pd.read_csv(IN_DIR/'train.csv')
test_df = pd.read_csv(IN_DIR/'test.csv')
required_cols = {'source_index','feedback_text','clean_text',LABEL_COL}
if not required_cols.issubset(train_df.columns) or not required_cols.issubset(test_df.columns):
    raise ValueError(f'Train/test phải chứa các cột: {required_cols}')
if train_df['clean_text'].isna().any() or test_df['clean_text'].isna().any():
    raise ValueError('Phát hiện văn bản đã làm sạch bị thiếu')
assert not set(train_df['source_index']) & set(test_df['source_index']), 'Lặp source_index giữa train/test'
assert not set(train_df['clean_text']) & set(test_df['clean_text']), 'Lặp clean_text giữa train/test'
assert train_df[LABEL_COL].notna().all() and test_df[LABEL_COL].notna().all()
X_train_text, X_test_text = train_df['clean_text'], test_df['clean_text']
y_train, y_test = train_df[LABEL_COL], test_df[LABEL_COL]
class_counts = y_train.value_counts()
if class_counts.min() < 3:
    raise ValueError('Cần >=3 mẫu cho mỗi lớp trong train để chạy StratifiedKFold 3 folds')
LABELS = sorted(set(y_train) | set(y_test))
print('Train:',len(train_df),'Test:',len(test_df),'| Labels:',LABELS)
display(pd.DataFrame({'train':y_train.value_counts(), 'test':y_test.value_counts()}).fillna(0).astype(int))


Train: 978 Test: 245 | Labels: ['negative', 'neutral', 'positive']


,train,test
sentiment_label,,
positive,391,98
neutral,294,74
negative,293,73


## 3. Baseline và kiểm chứng các ma trận đã xuất
Dùng BoW Unigram + MultinomialNB(alpha=1.0) làm baseline chính theo yêu cầu giảng viên.
Thêm mô hình Dummy dự đoán lớp phổ biến nhất để kiểm tra mô hình có vượt cách dự đoán đơn giản hay không.

In [3]:
X_train_bow = load_npz(IN_DIR/'BoW_Unigram_train.npz')
X_test_bow = load_npz(IN_DIR/'BoW_Unigram_test.npz')
assert X_train_bow.shape[0] == len(y_train) and X_test_bow.shape[0] == len(y_test)

def evaluate(y_true,y_pred):
    return {
        'accuracy': float(accuracy_score(y_true,y_pred)),
        'precision_macro': float(precision_score(y_true,y_pred,average='macro',zero_division=0)),
        'recall_macro': float(recall_score(y_true,y_pred,average='macro',zero_division=0)),
        'f1_macro': float(f1_score(y_true,y_pred,average='macro',zero_division=0)),
        'f1_weighted': float(f1_score(y_true,y_pred,average='weighted',zero_division=0))
    }

test_metrics, prediction_frames, cv_records = [], [], []
def save_predictions(model_id,pred,kind='model'):
    part = test_df[['source_index','feedback_text','clean_text',LABEL_COL]].copy()
    part['model_id'] = model_id
    part['predicted_label'] = pred
    part['is_correct'] = (part[LABEL_COL].to_numpy() == np.asarray(pred))
    part['model_kind'] = kind
    prediction_frames.append(part)

for model_id, estimator in [('Dummy_Majority',DummyClassifier(strategy='most_frequent')),
                            ('Baseline_BoW_NB',MultinomialNB(alpha=1.0))]:
    estimator.fit(X_train_bow,y_train)
    pred = estimator.predict(X_test_bow)
    test_metrics.append({'model_id':model_id,'vectorizer':'BoW_Unigram',
                         'algorithm':'Dummy' if model_id.startswith('Dummy') else 'NaiveBayes',
                         'hyperparameters':'default' if model_id.startswith('Dummy') else 'alpha=1',
                         'cv_f1_macro':float('nan'),**evaluate(y_test,pred)})
    save_predictions(model_id,pred,kind='reference' if model_id.startswith('Dummy') else 'baseline')
    print(model_id,'Accuracy=',round(accuracy_score(y_test,pred),4),'Macro F1=',round(f1_score(y_test,pred,average='macro'),4))


Dummy_Majority Accuracy= 0.4 Macro F1= 0.1905
Baseline_BoW_NB Accuracy= 0.3878 Macro F1= 0.3748


## 4. Thiết kế thử nghiệm (CV trên train)
So sánh 4 cách biểu diễn (BoW/TF-IDF, unigram/(uni+bi)) với 2 thuật toán (Naive Bayes, K-NN).
- Với Naive Bayes: alpha = [0.5, 1, 2].
- Với K-NN: k = [3, 5, 7], khoảng cách cosine trên dữ liệu văn bản thưa.
- scoring='f1_macro', StratifiedKFold 3 folds, random_state=42.
Số liệu test không được dùng để tìm k, alpha hoặc tinh chỉnh sau khi nhìn thấy kết quả.

In [4]:
vectorizer_specs = config['vectorizers']
cv = StratifiedKFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE)
candidates = []
cv_detail_records = []
for cfg_name,spec in vectorizer_specs.items():
    for alg_name in ('NaiveBayes','KNN'):
        vectorizer_class = CountVectorizer if spec['kind']=='BoW' else TfidfVectorizer
        vec = vectorizer_class(
            ngram_range=tuple(spec['ngram_range']), min_df=spec['min_df'],
            max_df=spec['max_df'], max_features=spec['max_features']
        )
        if alg_name=='NaiveBayes':
            est = MultinomialNB()
            param_grid = {'model__alpha':[0.5,1.0,2.0]}
        else:
            est = KNeighborsClassifier(metric='cosine',algorithm='brute',n_jobs=1)
            param_grid = {'model__n_neighbors':[3,5,7]}
        pipe = Pipeline([('vectorizer',vec),('model',est)])
        grid = GridSearchCV(pipe,param_grid=param_grid,cv=cv,scoring='f1_macro',n_jobs=1,refit=True)
        grid.fit(X_train_text,y_train)
        model_id = f'{cfg_name}__{alg_name}'
        # Lưu toàn bộ lượt thử alpha/k để phân tích ảnh hưởng siêu tham số.
        for param, mean, std, rank in zip(
            grid.cv_results_['params'],
            grid.cv_results_['mean_test_score'],
            grid.cv_results_['std_test_score'],
            grid.cv_results_['rank_test_score']
        ):
            cv_detail_records.append({
                'model_id':model_id, 'vectorizer':cfg_name, 'algorithm':alg_name,
                'hyperparameters':json.dumps(param,ensure_ascii=False),
                'cv_f1_macro_mean':float(mean), 'cv_f1_macro_std':float(std),
                'rank_in_configuration':int(rank),
                'selected_param_for_configuration':bool(param==grid.best_params_)
            })
        cv_score = float(grid.best_score_)
        cv_records.append({'model_id':model_id,'vectorizer':cfg_name,'algorithm':alg_name,
                           'cv_f1_macro':cv_score,'cv_f1_macro_std':float(grid.cv_results_['std_test_score'][grid.best_index_]),'best_params':json.dumps(grid.best_params_,ensure_ascii=False)})
        candidates.append((cv_score,model_id,grid.best_estimator_,cfg_name,alg_name,grid.best_params_))
        print(f'{model_id}: CV F1 macro={cv_score:.4f} | {grid.best_params_}')
cv_results = pd.DataFrame(cv_records).sort_values('cv_f1_macro',ascending=False).reset_index(drop=True)
cv_results.to_csv(OUT_DIR/'model_cv_results.csv',index=False,encoding='utf-8-sig')
display(cv_results)

cv_details_df = pd.DataFrame(cv_detail_records).sort_values(['model_id','cv_f1_macro_mean'],ascending=[True,False])
cv_details_df.to_csv(OUT_DIR/'cv_hyperparameter_details.csv',index=False,encoding='utf-8-sig')
print('Đã xuất',len(cv_details_df),'lượt thử alpha/k để giải thích kết quả CV.')

# Tổng hợp độ nhạy tham số (trên CV, KHÔNG dùng test để chọn tham số).
param_effect_rows = []
for (cfg,algorithm),part in cv_details_df.groupby(['vectorizer','algorithm']):
    part = part.sort_values('cv_f1_macro_mean',ascending=False)
    best = part.iloc[0]
    worst = part.iloc[-1]
    param_key = 'model__alpha' if algorithm=='NaiveBayes' else 'model__n_neighbors'
    param_effect_rows.append({
        'vectorizer':cfg,
        'algorithm':algorithm,
        'hyperparameter':param_key,
        'best_value':json.loads(best['hyperparameters'])[param_key],
        'best_cv_f1_macro':float(best['cv_f1_macro_mean']),
        'worst_value':json.loads(worst['hyperparameters'])[param_key],
        'worst_cv_f1_macro':float(worst['cv_f1_macro_mean']),
        'cv_f1_span':float(best['cv_f1_macro_mean']-worst['cv_f1_macro_mean'])
    })
param_effect_df = pd.DataFrame(param_effect_rows).sort_values('best_cv_f1_macro',ascending=False)
param_effect_df.to_csv(OUT_DIR/'cv_parameter_effect.csv',index=False,encoding='utf-8-sig')
print('Ảnh hưởng tham số alpha hoặc k (độ chênh chỉ tính trên train-CV):')
display(param_effect_df.round(4))


BoW_Unigram__NaiveBayes: CV F1 macro=0.3637 | {'model__alpha': 0.5}
BoW_Unigram__KNN: CV F1 macro=0.3555 | {'model__n_neighbors': 5}
BoW_Unigram_Bigram__NaiveBayes: CV F1 macro=0.3569 | {'model__alpha': 0.5}
BoW_Unigram_Bigram__KNN: CV F1 macro=0.3433 | {'model__n_neighbors': 7}
TFIDF_Unigram__NaiveBayes: CV F1 macro=0.3374 | {'model__alpha': 0.5}
TFIDF_Unigram__KNN: CV F1 macro=0.3647 | {'model__n_neighbors': 5}
TFIDF_Unigram_Bigram__NaiveBayes: CV F1 macro=0.2852 | {'model__alpha': 0.5}
TFIDF_Unigram_Bigram__KNN: CV F1 macro=0.3490 | {'model__n_neighbors': 3}


,model_id,vectorizer,algorithm,cv_f1_macro,cv_f1_macro_std,best_params
0,TFIDF_Unigram__KNN,TFIDF_Unigram,KNN,0.364671,0.008816,"{""model__n_neighbors"": 5}"
1,BoW_Unigram__NaiveBayes,BoW_Unigram,NaiveBayes,0.363663,0.014798,"{""model__alpha"": 0.5}"
2,BoW_Unigram_Bigram__NaiveBayes,BoW_Unigram_Bigram,NaiveBayes,0.356859,0.033167,"{""model__alpha"": 0.5}"
3,BoW_Unigram__KNN,BoW_Unigram,KNN,0.355485,0.025013,"{""model__n_neighbors"": 5}"
4,TFIDF_Unigram_Bigram__KNN,TFIDF_Unigram_Bigram,KNN,0.348973,0.012482,"{""model__n_neighbors"": 3}"
5,BoW_Unigram_Bigram__KNN,BoW_Unigram_Bigram,KNN,0.343315,0.013606,"{""model__n_neighbors"": 7}"
6,TFIDF_Unigram__NaiveBayes,TFIDF_Unigram,NaiveBayes,0.337365,0.034537,"{""model__alpha"": 0.5}"
7,TFIDF_Unigram_Bigram__NaiveBayes,TFIDF_Unigram_Bigram,NaiveBayes,0.285182,0.022327,"{""model__alpha"": 0.5}"


Đã xuất 24 lượt thử alpha/k để giải thích kết quả CV.
Ảnh hưởng tham số alpha hoặc k (độ chênh chỉ tính trên train-CV):


,vectorizer,algorithm,hyperparameter,best_value,best_cv_f1_macro,worst_value,worst_cv_f1_macro,cv_f1_span
4,TFIDF_Unigram,KNN,model__n_neighbors,5.0,0.3647,3.0,0.3489,0.0158
1,BoW_Unigram,NaiveBayes,model__alpha,0.5,0.3637,2.0,0.3564,0.0072
3,BoW_Unigram_Bigram,NaiveBayes,model__alpha,0.5,0.3569,2.0,0.3390,0.0179
0,BoW_Unigram,KNN,model__n_neighbors,5.0,0.3555,7.0,0.3413,0.0141
6,TFIDF_Unigram_Bigram,KNN,model__n_neighbors,3.0,0.3490,7.0,0.3445,0.0044
2,BoW_Unigram_Bigram,KNN,model__n_neighbors,7.0,0.3433,3.0,0.3374,0.0060
5,TFIDF_Unigram,NaiveBayes,model__alpha,0.5,0.3374,2.0,0.2834,0.0539
7,TFIDF_Unigram_Bigram,NaiveBayes,model__alpha,0.5,0.2852,2.0,0.2048,0.0804


## 5. Đánh giá sau khi cố định cấu hình bằng CV
Mỗi mô hình đã được chọn tham số trên train/CV. Bây giờ mới dự đoán tập test cố định.
Bảng này phục vụ so sánh thuật toán và cách biểu diễn. Mô hình đề xuất được chọn theo điểm CV, không chọn lại dựa vào điểm test.

In [5]:
best_score,best_model_id,best_pipeline,*_ = max(candidates,key=lambda x:x[0])
for cv_score,model_id,estimator,cfg_name,alg_name,best_params in candidates:
    preds = estimator.predict(X_test_text)
    test_metrics.append({
        'model_id':model_id,'vectorizer':cfg_name,'algorithm':alg_name,
        'hyperparameters':json.dumps(best_params,ensure_ascii=False),
        'cv_f1_macro':cv_score,**evaluate(y_test,preds)
    })
    save_predictions(model_id,preds,'cv_tuned')
    joblib.dump(estimator,MODEL_DIR/f'{model_id}.joblib')
metrics_df = pd.DataFrame(test_metrics)
metrics_df.to_csv(OUT_DIR/'test_metrics.csv',index=False,encoding='utf-8-sig')
preds_df = pd.concat(prediction_frames,ignore_index=True)
preds_df.to_csv(OUT_DIR/'test_predictions.csv',index=False,encoding='utf-8-sig')
joblib.dump(best_pipeline,OUT_DIR/'best_model.joblib')
(OUT_DIR/'best_model.json').write_text(json.dumps({
    'model_id':best_model_id,'selection_metric':'cv_f1_macro',
    'cv_f1_macro':best_score,'test_size':len(test_df),
    'input_requirement':'Văn bản mới phải được làm sạch bằng src.text_utils.clean_text trước khi predict',
    'random_state':RANDOM_STATE
},indent=2,ensure_ascii=False),encoding='utf-8')
print('Mô hình được chọn từ CV:',best_model_id,'| CV F1 macro:',round(best_score,4))
print('Đã lưu:',OUT_DIR)
display(metrics_df.sort_values('f1_macro',ascending=False).round(4))

Mô hình được chọn từ CV: TFIDF_Unigram__KNN | CV F1 macro: 0.3647
Đã lưu: D:\Nam3\MachineLearning\Nhom-7_Phan_loai_cam_xuc_tu_phan_hoi_cua_sinh_vien-main\results\training


,model_id,vectorizer,algorithm,hyperparameters,cv_f1_macro,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted
1,Baseline_BoW_NB,BoW_Unigram,NaiveBayes,alpha=1,NaN,0.3878,0.3764,0.3751,0.3748,0.3846
2,BoW_Unigram__NaiveBayes,BoW_Unigram,NaiveBayes,"{""model__alpha"": 0.5}",0.3637,0.3837,0.3727,0.3706,0.3702,0.3800
6,TFIDF_Unigram__NaiveBayes,TFIDF_Unigram,NaiveBayes,"{""model__alpha"": 0.5}",0.3374,0.3918,0.3801,0.3683,0.3614,0.3744
3,BoW_Unigram__KNN,BoW_Unigram,KNN,"{""model__n_neighbors"": 5}",0.3555,0.3592,0.3670,0.3618,0.3588,0.3618
4,BoW_Unigram_Bigram__NaiveBayes,BoW_Unigram_Bigram,NaiveBayes,"{""model__alpha"": 0.5}",0.3569,0.3673,0.3557,0.3481,0.3436,0.3549
7,TFIDF_Unigram__KNN,TFIDF_Unigram,KNN,"{""model__n_neighbors"": 5}",0.3647,0.3388,0.3431,0.3427,0.3369,0.3384
8,TFIDF_Unigram_Bigram__NaiveBayes,TFIDF_Unigram_Bigram,NaiveBayes,"{""model__alpha"": 0.5}",0.2852,0.4041,0.4019,0.3639,0.3316,0.3520
9,TFIDF_Unigram_Bigram__KNN,TFIDF_Unigram_Bigram,KNN,"{""model__n_neighbors"": 3}",0.3490,0.3347,0.3479,0.3360,0.3316,0.3349
5,BoW_Unigram_Bigram__KNN,BoW_Unigram_Bigram,KNN,"{""model__n_neighbors"": 7}",0.3433,0.3347,0.3270,0.3275,0.3271,0.3341
0,Dummy_Majority,BoW_Unigram,Dummy,default,NaN,0.4000,0.1333,0.3333,0.1905,0.2286


## 5.1. Nhận xét số liệu CV, baseline và test


In [6]:
from IPython.display import Markdown

ranked_cv = cv_results.sort_values('cv_f1_macro',ascending=False).reset_index(drop=True)
cv_top = float(ranked_cv.iloc[0]['cv_f1_macro'])
cv_second = float(ranked_cv.iloc[1]['cv_f1_macro']) if len(ranked_cv)>1 else float('nan')
chosen_test = metrics_df.loc[metrics_df.model_id==best_model_id].iloc[0]
base_test = metrics_df.loc[metrics_df.model_id=='Baseline_BoW_NB'].iloc[0]
dummy_test = metrics_df.loc[metrics_df.model_id=='Dummy_Majority'].iloc[0]
train_notes = f"""# Diễn giải kết quả thực nghiệm – Thành viên 3

- Đã thử **{len(cv_results)}** tổ hợp biểu diễn–thuật toán và **{len(cv_details_df)}** lượt cấu hình tham số bằng Stratified 3-fold CV **chỉ trên train** (tiêu chí F1-macro).
- Điểm CV tốt nhất thuộc **{best_model_id}**: **{cv_top:.4f}**; hạng hai **{ranked_cv.iloc[1]['model_id']}**: **{cv_second:.4f}**. Mức chênh CV chỉ **{cv_top-cv_second:.4f}**; một chênh lệch rất nhỏ chưa đủ để khẳng định một thuật toán tốt hơn ổn định.
- Sau khi chọn theo CV, mô hình được đánh giá một lần trên tập test cố định: **Accuracy={chosen_test['accuracy']:.4f}**, **Macro F1={chosen_test['f1_macro']:.4f}**; baseline BoW+NB **Macro F1={base_test['f1_macro']:.4f}**. Mức chênh (mô hình chọn – baseline) là **{chosen_test['f1_macro']-base_test['f1_macro']:+.4f}**.
- Dummy majority có **Accuracy={dummy_test['accuracy']:.4f}**, **Macro F1={dummy_test['f1_macro']:.4f}**. Chỉ số Accuracy một mình có thể đánh lừa khi dự đoán dồn vào lớp đông nhất.
- Việc mô hình CV tốt nhất có điểm test thấp hơn baseline **không chứng minh thuật toán bị lỗi**. Cần xem xét biến thiên do cách chia dữ liệu, chất lượng nhãn/văn bản và đặc trưng; **không chọn lại theo test** sau khi xem kết quả.
- Lịch sử 24 lượt thử nằm tại `results/training/cv_hyperparameter_details.csv`; bảng độ nhạy alpha/k nằm tại `results/training/cv_parameter_effect.csv`. Đầu ra chuyển cho Thành viên 4 là `test_metrics.csv`, `test_predictions.csv`, `best_model.json`.
"""
(OUT_DIR/'training_interpretation.md').write_text(train_notes,encoding='utf-8')
display(Markdown(train_notes))


# Diễn giải kết quả thực nghiệm – Thành viên 3

- Đã thử **8** tổ hợp biểu diễn–thuật toán và **24** lượt cấu hình tham số bằng Stratified 3-fold CV **chỉ trên train** (tiêu chí F1-macro).
- Điểm CV tốt nhất thuộc **TFIDF_Unigram__KNN**: **0.3647**; hạng hai **BoW_Unigram__NaiveBayes**: **0.3637**. Mức chênh CV chỉ **0.0010**; một chênh lệch rất nhỏ chưa đủ để khẳng định một thuật toán tốt hơn ổn định.
- Sau khi chọn theo CV, mô hình được đánh giá một lần trên tập test cố định: **Accuracy=0.3388**, **Macro F1=0.3369**; baseline BoW+NB **Macro F1=0.3748**. Mức chênh (mô hình chọn – baseline) là **-0.0379**.
- Dummy majority có **Accuracy=0.4000**, **Macro F1=0.1905**. Chỉ số Accuracy một mình có thể đánh lừa khi dự đoán dồn vào lớp đông nhất.
- Việc mô hình CV tốt nhất có điểm test thấp hơn baseline **không chứng minh thuật toán bị lỗi**. Cần xem xét biến thiên do cách chia dữ liệu, chất lượng nhãn/văn bản và đặc trưng; **không chọn lại theo test** sau khi xem kết quả.
- Lịch sử 24 lượt thử nằm tại `results/training/cv_hyperparameter_details.csv`; bảng độ nhạy alpha/k nằm tại `results/training/cv_parameter_effect.csv`. Đầu ra chuyển cho Thành viên 4 là `test_metrics.csv`, `test_predictions.csv`, `best_model.json`.


## 6. Kiểm tra và bàn giao cho Thành viên 4
File 04 đọc results/training/test_metrics.csv + test_predictions.csv + best_model.json để dựng
bảng so sánh, báo cáo phân loại, Confusion Matrix và phân tích các mẫu sai.
Các con số chỉ có giá trị sau khi chạy thực tế với dataset được chỉ định.

In [7]:
assert metrics_df['model_id'].is_unique
assert preds_df.groupby('model_id').size().eq(len(test_df)).all()
assert not preds_df[['source_index','model_id']].duplicated().any()
print('Kiểm tra đầu ra thành công. File 04 có thể thực hiện đánh giá.')


Kiểm tra đầu ra thành công. File 04 có thể thực hiện đánh giá.
